In [16]:
%pip install ete3
import pandas as pd
from ete3 import Tree
import re

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 23.3.1 -> 25.0.1
[notice] To update, run: /Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [17]:
#with open("3oh_only.tree", 'r') as f:
with open("fa_only.tree", 'r') as f:
    newick_str = f.read()

In [18]:
open_count = newick_str.count('(')
close_count = newick_str.count(')')


if open_count > close_count:
    newick_str += ')' * (open_count - close_count)


if not newick_str.endswith(';'):
    newick_str += ';'


try:
    tree = Tree(newick_str)
except Exception as e:
    print(f"Error parsing tree: {e}")

    pattern = r'([A-Za-z0-9_-]+):([-]?[0-9]+\.[0-9]+)'
    node_names = re.findall(pattern, newick_str)
    if node_names:
        print(f"Found {len(node_names)} nodes using regex pattern.")

        simple_newick = "(" + ",".join([f"{name}:{dist}" for name, dist in node_names]) + ");"
        tree = Tree(simple_newick)
    else:
        raise Exception("Could not parse the tree in any way")

def find_closest_node(node, all_leaves):
    distances = []
    node_name = node.name
    
    for leaf in all_leaves:
        if leaf.name != node_name:
            dist = node.get_distance(leaf)
            distances.append((leaf.name, dist))
    

    distances.sort(key=lambda x: x[1])
    

    if distances:
        return distances[0]
    return None


leaves = tree.get_leaves()


results = []


closest_match = {}
for leaf in leaves:
    closest = find_closest_node(leaf, leaves)
    if closest:
        closest_name, distance = closest
        results.append({
            'Node': leaf.name.split('--')[0],
            'Closest Match': closest_name.split('--')[0],
            'Distance': distance
        })
        closest_match[leaf.name.split('--')[0]] = closest_name.split('--')[0]


df = pd.DataFrame(results)
print(df.to_string(index=False))


df.to_csv('closest_matches.csv', index=False)
print("\nResults saved to 'closest_matches.csv'")

                            Node              Closest Match  Distance
                COND0000291-lptA           COND0000336-dptA   0.58116
                COND0000336-dptA     COND0000439-AHH53506.1   0.57471
          COND0000439-AHH53506.1           COND0000336-dptA   0.57471
                COND0001968-cdeI           COND0001448-mlcK   0.60112
                COND0002430-gauA           COND0001448-mlcK   0.60688
           COND0002367-SCAB_3321           COND0001448-mlcK   0.61535
                COND0000354-pstB           COND0000379-lpmB   0.42396
                COND0000379-lpmB           COND0000354-pstB   0.42396
                COND0001984-speA           COND0000354-pstB   0.57123
                COND0001448-mlcK           COND0000354-pstB   0.56869
             COND0000315-SCO3230        COND0001370-cda2PSI   0.12027
             COND0001370-cda2PSI        COND0000315-SCO3230   0.12027
                COND0000305-arfA           COND0001980-lokA   0.02018
                COND

In [19]:
open_count = newick_str.count('(')
close_count = newick_str.count(')')


if open_count > close_count:
    newick_str += ')' * (open_count - close_count)


if not newick_str.endswith(';'):
    newick_str += ';'


try:
    tree = Tree(newick_str)
except Exception as e:
    print(f"Error parsing tree: {e}")

    pattern = r'([A-Za-z0-9_-]+):([-]?[0-9]+\.[0-9]+)'
    node_names = re.findall(pattern, newick_str)
    if node_names:
        print(f"Found {len(node_names)} nodes using regex pattern.")

        simple_newick = "(" + ",".join([f"{name}:{dist}" for name, dist in node_names]) + ");"
        tree = Tree(simple_newick)
    else:
        raise Exception("Could not parse the tree in any way")

def find_closest_node(node, all_leaves):
    distances = []
    node_name = node.name
    
    for leaf in all_leaves:
        if leaf.name != node_name:
            dist = node.get_distance(leaf)
            distances.append((leaf.name, dist))
    

    distances.sort(key=lambda x: x[1])
    

    if distances:
        return distances[0]
    return None


leaves = tree.get_leaves()


results = []


closest_match = {}
for leaf in leaves:
    closest = find_closest_node(leaf, leaves)
    if closest:
        closest_name, distance = closest
        results.append({
            'Node': leaf.name,
            'Closest Match': closest_name,
            'Distance': distance
        })
        closest_match[leaf.name] = closest_name


df = pd.DataFrame(results)
# print(df.to_string(index=False))


df.to_csv('closest_matches.csv', index=False)
print("\nResults saved to 'closest_matches.csv'")

ar_confusion_matrix = [[0,0],[0,0]]
ar_confusion_ref = {'AR': 0, 'FA': 1}

len_confusion_matrix = [[0,0,0,0],[0,0,0,0],[0,0,0,0],[0,0,0,0]]
len_confusion_ref = {'AR': 0, 'SCFA': 1, 'MCFA': 2, 'LCFA': 3}

boh_confusion_matrix = [[0,0], [0,0]]
boh_confusion_ref = {'3OH': 1, 'no_3OH': 0}

for i in range(len(df)):
    node = df.iloc[i]["Node"].split("--")
    pred = df.iloc[i]["Closest Match"].split("--")

    # print(node[1][-2:], pred[1][-2:])
    ar_confusion_matrix[ar_confusion_ref[node[1][-2:]]][ar_confusion_ref[pred[1][-2:]]] += 1
    
    len_confusion_matrix[len_confusion_ref[node[1]]][len_confusion_ref[pred[1]]] += 1

    if len(node) == 3 and len(pred) == 3:
        boh_confusion_matrix[boh_confusion_ref[node[2]]][boh_confusion_ref[pred[2]]] += 1

print("AR-FA")
print(ar_confusion_matrix[0])
print(ar_confusion_matrix[1])

correct = 0
total = 0
for i in range(len(ar_confusion_matrix)):
    correct += ar_confusion_matrix[i][i]
    total += sum(ar_confusion_matrix[i])
print(correct/total)


print('\nAR+len')
print(len_confusion_matrix[0])
print(len_confusion_matrix[1])
print(len_confusion_matrix[2])
print(len_confusion_matrix[3])

correct = 0
total = 0
for i in range(len(len_confusion_matrix)):
    correct += len_confusion_matrix[i][i]
    total += sum(len_confusion_matrix[i])
print(correct/total)


print('\n3OH')
print(boh_confusion_matrix[0])
print(boh_confusion_matrix[1])
correct = 0
total = 0
for i in range(len(boh_confusion_matrix)):
    correct += boh_confusion_matrix[i][i]
    total += sum(boh_confusion_matrix[i])
print(correct/total)






Results saved to 'closest_matches.csv'
AR-FA
[0, 0]
[0, 133]
1.0

AR+len
[0, 0, 0, 0]
[0, 20, 5, 8]
[0, 8, 13, 8]
[0, 8, 3, 60]
0.6992481203007519

3OH
[46, 12]
[12, 41]
0.7837837837837838


In [ ]:
3OH
[54, 12]
[13, 41]
0.7916666666666666

AR-FA
[66, 1]
[23, 110]
0.88

AR+len
[0, 0, 0, 0]
[0, 20, 5, 8]
[0, 8, 13, 8]
[0, 8, 3, 60]
0.6992481203007519


In [32]:
pl_df = pd.read_csv("condensation_starter_dataset_philmuslab.csv")

cond_to_category = dict(zip(pl_df['cond_accession'], pl_df['category']))

cond_to_category = {k: v for k, v in zip(pl_df['cond_accession'], pl_df['category']) 
                   if pd.notna(k) and pd.notna(v)}


boh_df = pd.read_csv("csv/condensation_starter.csv")
cond_to_boh = dict(zip(boh_df['cond_accession'], boh_df['3OH']))
cond_to_boh = {k: v for k, v in zip(boh_df['cond_accession'], boh_df['3OH']) 
                   if pd.notna(k) and pd.notna(v)}

In [33]:
total = 0
same = 0

for cond, closest in closest_match.items():
    print(cond, closest)
    if cond in cond_to_boh and closest in cond_to_boh:
        total += 1
        if cond_to_boh[cond] == cond_to_boh[closest]:
            same += 1

same, total, same/total

COND0000291-lptA COND0000336-dptA
COND0000336-dptA COND0000439-AHH53506.1
COND0000439-AHH53506.1 COND0000336-dptA
COND0000315-SCO3230 COND0001370-cda2PSI
COND0001370-cda2PSI COND0000315-SCO3230
COND0000354-pstB COND0000379-lpmB
COND0000379-lpmB COND0000354-pstB
COND0001984-speA COND0000354-pstB
COND0001448-mlcK COND0000354-pstB
COND0001968-cdeI COND0001448-mlcK
COND0002430-gauA COND0001448-mlcK
COND0002367-SCAB_3321 COND0001448-mlcK
COND0000343-ACY02015.1 COND0001502-AHA_2476
COND0001502-AHA_2476 COND0000343-ACY02015.1
COND0002528-AvCA_21160 COND0002528-AvCA_21190
COND0002528-AvCA_21190 COND0002528-AvCA_21160
COND0002491-TW75_20440 COND0001695-natB
COND0001437-obaI COND0001695-natB
COND0000454-vabF COND0002415-DSB67_23665
COND0002415-DSB67_23665 COND0000454-vabF
COND0002413-DJ58_4257 COND0002414-JJO56_07700
COND0002414-JJO56_07700 COND0002413-DJ58_4257
COND0002680-CV_2233 COND0002413-DJ58_4257
COND0002473-B090_RS0102665 COND0001695-natB
COND0002685-ROTMU0001_0989 COND0001695-natB
COND0

(73, 85, 0.8588235294117647)

In [19]:
total = 0
same = 0 

ar_t = 0
ar_s = 0

len_t = 0
len_s = 0

for cond, closest in closest_match.items():
    if cond in cond_to_category and closest in cond_to_category:
        total += 1
        if cond_to_category[cond] == 'AR':
            ar_t += 1
            if cond_to_category[cond] == cond_to_category[closest]:
                ar_s += 1
        else:
            len_t += 1
            if cond_to_category[cond] == cond_to_category[closest]:
                len_s += 1

        if cond_to_category[cond] == cond_to_category[closest]:
            same += 1

same, total, same/total, len_t, len_s, len_s/len_t

(152, 201, 0.7562189054726368, 134, 86, 0.6417910447761194)

In [20]:
total = 0
same = 0 

for cond, closest in closest_match.items():
    if cond in cond_to_category and closest in cond_to_category:
        total += 1
        if cond_to_category[cond][2:] == cond_to_category[closest][2:]:
            same += 1

same, total, same/total, len_t, len_s, len_s/len_t

(176, 201, 0.8756218905472637, 134, 86, 0.6417910447761194)

## NEW TREES

In [20]:
import pandas as pd

df = pd.read_csv('csv/condensation_starter.csv')
cat = {k: v for k, v in zip(df['cond_accession'], df['category']) if pd.notna(k) and pd.notna(v)}
boh = {k: v for k, v in zip(df['cond_accession'], df['3OH']) if pd.notna(k) and pd.notna(v)}
leng = {k: v for k, v in zip(df['cond_accession'], df['size_carbons']) if pd.notna(k) and pd.notna(v)}


with open('fasta/c_start_seq_ol.fasta', 'r') as rf:
    with open('fasta/c_start_seq_ol_labeled.fasta', 'w') as wf:
        r = rf.read()
        for acc in df['cond_accession']:
            replacement_str = acc
            if acc in cat:
                replacement_str += '--' + cat[acc]
                if cat[acc] == 'SCFA' and acc in leng:
                    if int(leng[acc]) == 2:
                        replacement_str += ('[Ac]')
                        if acc in boh:
                            print(acc, 'should be ac')
            if acc in boh:
                if boh[acc]:
                    replacement_str += '--' + '3OH'
                else:
                    replacement_str += '--' + 'no_3OH'
            else:
                if acc in cat:     
                    if cat[acc] != 'AR':
                        if acc in leng:
                            print(acc, cat[acc], leng[acc]) 
                        else:
                            print(acc, cat[acc])
            if replacement_str == acc:
                replacement_str += '--UNKNOWN'
            r = r.replace(acc, replacement_str)
        wf.write(r)



COND0000346-epxD SCFA 2.0
COND0000349-SACE_3035 SCFA 2.0
COND0000350-ETU_000009 FA
COND0000363-mps1 LCFA
COND0000364-mps1 LCFA
COND0001033-ERIC2_c18170 FA
COND0001211-AMYAL_RS0130210 SCFA 2.0
COND0001414-griA SCFA 2.0
COND0001758-RBRH_RS12370 SCFA 2.0
COND0001767-ctg1_orf20 SCFA 2.0
COND0002125-frsD SCFA 2.0
COND0002314-DMA15_34330 SCFA 2.0
COND0002466-CGZ69_03795 SCFA 2.0
COND0002467-YWY_RS21605 SCFA 2.0
COND0002581-QKF54435.2 SCFA


In [ ]:
def filter_fasta_file(input_file, output_file):
    """
    Filter a FASTA file to remove entries labeled as 'UNKNOWN'.
    
    Args:
        input_file (str): Path to the input FASTA file
        output_file (str): Path to the output filtered FASTA file
    """
    with open(input_file, 'r') as infile, open(output_file, 'w') as outfile:
        keep_entry = False
        current_header = ""
        current_sequence = []
        
        for line in infile:
            line = line.rstrip()
            
            if line.startswith('>'):
                if keep_entry and current_header:
                    outfile.write(current_header + '\n')
                    outfile.write('\n'.join(current_sequence) + '\n')

                current_header = line
                current_sequence = []
                
                keep_entry = '--UNKNOWN' not in line and line[-4:] != '--FA'
            else:
                current_sequence.append(line)

        if keep_entry and current_header:
            outfile.write(current_header + '\n')
            outfile.write('\n'.join(current_sequence) + '\n')

filter_fasta_file('fasta/c_start_seq_ol_labeled.fasta', 'fasta/c_start_seq_ol_labeled_known.fasta')

In [ ]:
def filter_fasta_file(input_file, output_file):
    """
    Filter a FASTA file to remove entries labeled as 'UNKNOWN'.
    
    Args:
        input_file (str): Path to the input FASTA file
        output_file (str): Path to the output filtered FASTA file
    """
    with open(input_file, 'r') as infile, open(output_file, 'w') as outfile:
        keep_entry = False
        current_header = ""
        current_sequence = []
        
        for line in infile:
            line = line.rstrip()
            
            if line.startswith('>'):
                if keep_entry and current_header:
                    outfile.write(current_header + '\n')
                    outfile.write('\n'.join(current_sequence) + '\n')
                

                current_header = line
                current_sequence = []
                
                keep_entry = '--UNKNOWN' not in line and '3OH' in line
            else:
                current_sequence.append(line)
        
        if keep_entry and current_header:
            outfile.write(current_header + '\n')
            outfile.write('\n'.join(current_sequence) + '\n')

filter_fasta_file('fasta/c_start_seq_ol_labeled.fasta', 'fasta/c_start_seq_3OH_only.fasta')